# 🛒 Customer Behavior Analysis — E-Commerce Dataset
### IBM SkillsBuild Data Analytics with AI Internship Project

---

**Dataset:** `Ecommerce.csv` — 25,000 e-commerce session records  
**Objective:** Analyze customer behavior patterns, identify purchase drivers, segment customers, and build a predictive model for purchase conversion.  

---

## 📋 Table of Contents
1. [Environment Setup & Imports](#1)
2. [Data Loading & First Look](#2)
3. [Data Cleaning & Preprocessing](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Feature Engineering](#5)
6. [Customer Segmentation](#6)
7. [Purchase Prediction Model (AI)](#7)
8. [Insights & Business Recommendations](#8)

---
## 1. Environment Setup & Imports <a id='1'></a>

In [ ]:
# Standard library
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')

---
## 2. Data Loading & First Look <a id='2'></a>

In [ ]:
# Load dataset
df = pd.read_csv('Ecommerce.csv')

print(f'Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Column names and data types
print('=== Column Info ===')
df.info()

In [ ]:
# Statistical summary
df.describe(include='all').T

### 🗂️ Column Glossary

| Column | Description | Type |
|---|---|---|
| `customer_id` | Unique customer identifier | ID |
| `session_id` | Unique session identifier | ID |
| `visit_date` | Date of the session (DD-MM-YYYY) | Date |
| `device_type` | 0=Desktop, 1=Mobile, 2=Tablet | Categorical |
| `user_type` | 0=New User, 1=Returning User | Categorical |
| `marketing_channel` | 0=Organic, 1=Email, 2=Social, 3=Direct, 4=Paid, 5=Referral | Categorical |
| `product_id` | Product identifier | ID |
| `product_category` | 0=Electronics … 7=Other | Categorical |
| `unit_price` | Price per unit (₹) | Numeric |
| `quantity` | Units in session | Numeric |
| `discount_percent` | Discount applied (%) | Numeric |
| `discount_amount` | Discount in currency | Numeric |
| `revenue` | Total revenue from session | Numeric |
| `pages_viewed` | Number of pages visited | Numeric |
| `time_on_site_sec` | Session duration (seconds) | Numeric |
| `added_to_cart` | 1=Yes, 0=No | Binary |
| `purchased` | 1=Purchase made, 0=No | Binary (Target) |
| `cart_abandoned` | 1=Cart abandoned, 0=No | Binary |
| `rating` | Product rating (1–5) | Numeric |
| `review_helpful_votes` | Helpful votes on review | Numeric |
| `payment_method` | 0=COD, 1=Credit, 2=Debit, 3=Wallet, 4=UPI, 5=Net Banking | Categorical |
| `visit_day` | Day of month | Numeric |
| `visit_month` | Month number | Numeric |
| `visit_weekday` | Day of week (0=Mon … 6=Sun) | Numeric |
| `visit_season` | 0=Winter, 1=Spring, 2=Summer, 3=Autumn | Categorical |
| `session_duration_bucket` | Very Short / Short / Long / Very Long | Categorical |
| `revenue_normalized` | Revenue scaled 0–1 | Numeric |
| `location` | Location code | Categorical |

---
## 3. Data Cleaning & Preprocessing <a id='3'></a>

In [ ]:
# ── 3.1 Missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_with_data = missing_df[missing_df['Missing Count'] > 0]

if missing_with_data.empty:
    print('✅ No missing values found — dataset is complete!')
else:
    print(missing_with_data)

In [ ]:
# ── 3.2 Duplicate rows ──────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'  Removed {duplicates} duplicates. New shape: {df.shape}')

In [ ]:
# ── 3.3 Parse date column ────────────────────────────────────────────────────
df['visit_date'] = pd.to_datetime(df['visit_date'], format='%d-%m-%Y')
print(f'Date range: {df["visit_date"].min().date()}  →  {df["visit_date"].max().date()}')

In [ ]:
# ── 3.4 Decode categorical columns ──────────────────────────────────────────
device_map      = {0: 'Desktop', 1: 'Mobile', 2: 'Tablet'}
user_map        = {0: 'New', 1: 'Returning'}
channel_map     = {0: 'Organic', 1: 'Email', 2: 'Social', 3: 'Direct', 4: 'Paid', 5: 'Referral'}
category_map    = {0: 'Electronics', 1: 'Clothing', 2: 'Home & Living', 3: 'Sports', 4: 'Beauty', 5: 'Books', 6: 'Toys', 7: 'Other'}
payment_map     = {0: 'COD', 1: 'Credit Card', 2: 'Debit Card', 3: 'Wallet', 4: 'UPI', 5: 'Net Banking'}
season_map      = {0: 'Winter', 1: 'Spring', 2: 'Summer', 3: 'Autumn'}
weekday_map     = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}

df['device_label']    = df['device_type'].map(device_map)
df['user_label']      = df['user_type'].map(user_map)
df['channel_label']   = df['marketing_channel'].map(channel_map)
df['category_label']  = df['product_category'].map(category_map)
df['payment_label']   = df['payment_method'].map(payment_map)
df['season_label']    = df['visit_season'].map(season_map)
df['weekday_label']   = df['visit_weekday'].map(weekday_map)

print('✅ Categorical columns decoded.')
print(f'\nPurchase rate overall: {df["purchased"].mean()*100:.2f}%')

In [ ]:
# ── 3.5 Outlier check on revenue ────────────────────────────────────────────
rev_non_zero = df[df['revenue'] > 0]['revenue']
q1, q3 = rev_non_zero.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 3 * iqr
outliers = (rev_non_zero > upper_fence).sum()
print(f'Revenue (non-zero) — Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}')
print(f'Upper fence (3×IQR): {upper_fence:.2f}')
print(f'Extreme outliers   : {outliers} sessions ({outliers/len(df)*100:.2f}%)')

---
## 4. Exploratory Data Analysis (EDA) <a id='4'></a>

### 4.1 Target Variable — Purchase Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['purchased'].value_counts()
colors = ['#e07b6a', '#5ba17c']
axes[0].bar(['Not Purchased (0)', 'Purchased (1)'], counts.values, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 150, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Purchase Count Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Sessions')

# Pie chart
axes[1].pie(counts.values, labels=['Not Purchased', 'Purchased'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Purchase Rate', fontweight='bold')

plt.suptitle('Target Variable: Purchase Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_purchase_distribution.png', bbox_inches='tight')
plt.show()
print(f'\nPurchased  : {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')
print(f'Not Bought : {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')

### 4.2 Device Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sessions by device
device_counts = df['device_label'].value_counts()
axes[0].bar(device_counts.index, device_counts.values,
            color=['#4c72b0', '#dd8452', '#55a868'], edgecolor='white')
axes[0].set_title('Sessions by Device Type', fontweight='bold')
axes[0].set_ylabel('Session Count')
for i, v in enumerate(device_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

# Purchase rate by device
purchase_rate_device = df.groupby('device_label')['purchased'].mean() * 100
purchase_rate_device = purchase_rate_device.sort_values(ascending=False)
bars = axes[1].bar(purchase_rate_device.index, purchase_rate_device.values,
                   color=['#4c72b0', '#dd8452', '#55a868'], edgecolor='white')
axes[1].set_title('Purchase Rate by Device Type', fontweight='bold')
axes[1].set_ylabel('Purchase Rate (%)')
axes[1].set_ylim(0, max(purchase_rate_device.values) * 1.2)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', fontweight='bold')

plt.suptitle('Device Type Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_device_analysis.png', bbox_inches='tight')
plt.show()

### 4.3 Marketing Channel Effectiveness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

channel_counts = df['channel_label'].value_counts()
palette = sns.color_palette('Set2', len(channel_counts))

# Session volume
axes[0].barh(channel_counts.index, channel_counts.values, color=palette)
axes[0].set_title('Sessions by Marketing Channel', fontweight='bold')
axes[0].set_xlabel('Session Count')
for i, v in enumerate(channel_counts.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center')

# Purchase rate
pr_channel = df.groupby('channel_label')['purchased'].mean().mul(100).sort_values(ascending=True)
axes[1].barh(pr_channel.index, pr_channel.values, color=palette)
axes[1].set_title('Purchase Rate by Marketing Channel', fontweight='bold')
axes[1].set_xlabel('Purchase Rate (%)')
for i, v in enumerate(pr_channel.values):
    axes[1].text(v + 0.1, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.suptitle('Marketing Channel Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_channel_analysis.png', bbox_inches='tight')
plt.show()

### 4.4 Product Category Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Revenue by category (purchased sessions only)
cat_revenue = df[df['purchased']==1].groupby('category_label')['revenue'].sum().sort_values(ascending=False)
colors_cat = sns.color_palette('tab10', len(cat_revenue))
axes[0].bar(cat_revenue.index, cat_revenue.values, color=colors_cat)
axes[0].set_title('Total Revenue by Product Category', fontweight='bold')
axes[0].set_ylabel('Revenue (₹)')
axes[0].set_xticklabels(cat_revenue.index, rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₹{x/1e6:.1f}M'))

# Purchase rate by category
pr_cat = df.groupby('category_label')['purchased'].mean().mul(100).sort_values(ascending=False)
axes[1].bar(pr_cat.index, pr_cat.values, color=colors_cat)
axes[1].set_title('Purchase Rate by Product Category', fontweight='bold')
axes[1].set_ylabel('Purchase Rate (%)')
axes[1].set_xticklabels(pr_cat.index, rotation=30, ha='right')
for i, v in enumerate(pr_cat.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Product Category Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_category_analysis.png', bbox_inches='tight')
plt.show()

### 4.5 Session Behavior — Pages Viewed & Time on Site

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pages viewed distribution by purchase
for label, group in df.groupby('purchased'):
    axes[0].hist(group['pages_viewed'], bins=25, alpha=0.65,
                 label='Purchased' if label==1 else 'Not Purchased', density=True)
axes[0].set_title('Pages Viewed Distribution', fontweight='bold')
axes[0].set_xlabel('Pages Viewed')
axes[0].set_ylabel('Density')
axes[0].legend()

# Time on site distribution
for label, group in df.groupby('purchased'):
    axes[1].hist(group['time_on_site_sec'] / 60, bins=30, alpha=0.65,
                 label='Purchased' if label==1 else 'Not Purchased', density=True)
axes[1].set_title('Time on Site Distribution', fontweight='bold')
axes[1].set_xlabel('Time on Site (minutes)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Session Behavior: Purchasers vs Non-Purchasers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_session_behavior.png', bbox_inches='tight')
plt.show()

### 4.6 Discount Impact on Purchases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg discount by purchase
avg_disc = df.groupby('purchased')['discount_percent'].mean()
labels = ['Not Purchased', 'Purchased']
axes[0].bar(labels, avg_disc.values, color=['#e07b6a', '#5ba17c'], width=0.4)
axes[0].set_title('Average Discount % by Purchase Outcome', fontweight='bold')
axes[0].set_ylabel('Avg Discount (%)')
for i, v in enumerate(avg_disc.values):
    axes[0].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold')

# Purchase rate across discount buckets
df['disc_bucket'] = pd.cut(df['discount_percent'], bins=[-1, 0, 10, 20, 30, 100],
                           labels=['No Discount', '1-10%', '11-20%', '21-30%', '>30%'])
pr_disc = df.groupby('disc_bucket', observed=True)['purchased'].mean().mul(100)
axes[1].bar(pr_disc.index.astype(str), pr_disc.values,
            color=sns.color_palette('Blues_d', len(pr_disc)))
axes[1].set_title('Purchase Rate by Discount Level', fontweight='bold')
axes[1].set_ylabel('Purchase Rate (%)')
axes[1].set_xlabel('Discount Bucket')
for i, v in enumerate(pr_disc.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Discount Impact on Purchase Behavior', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_discount_impact.png', bbox_inches='tight')
plt.show()

### 4.7 Revenue Trends Over Time

In [ ]:
# Monthly revenue trend
monthly = df[df['purchased']==1].copy()
monthly['year_month'] = monthly['visit_date'].dt.to_period('M')
monthly_rev = monthly.groupby('year_month')['revenue'].sum().reset_index()
monthly_rev['year_month_str'] = monthly_rev['year_month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Monthly revenue
axes[0].plot(monthly_rev['year_month_str'], monthly_rev['revenue'],
             marker='o', linewidth=2, color='#3b82d4', markersize=6)
axes[0].fill_between(range(len(monthly_rev)), monthly_rev['revenue'], alpha=0.15, color='#3b82d4')
axes[0].set_xticks(range(len(monthly_rev)))
axes[0].set_xticklabels(monthly_rev['year_month_str'], rotation=45, ha='right')
axes[0].set_title('Monthly Revenue Trend (2024)', fontweight='bold')
axes[0].set_ylabel('Total Revenue (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₹{x/1e6:.1f}M'))

# Sessions per month
monthly_sessions = df.groupby(df['visit_date'].dt.to_period('M')).size().reset_index()
monthly_sessions.columns = ['period', 'count']
monthly_sessions['period_str'] = monthly_sessions['period'].astype(str)
axes[1].bar(monthly_sessions['period_str'], monthly_sessions['count'],
            color='#5ba17c', edgecolor='white')
axes[1].set_xticklabels(monthly_sessions['period_str'], rotation=45, ha='right')
axes[1].set_title('Monthly Session Volume (2024)', fontweight='bold')
axes[1].set_ylabel('Number of Sessions')

plt.suptitle('Temporal Trends', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_temporal_trends.png', bbox_inches='tight')
plt.show()

### 4.8 Weekday & Seasonal Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weekday purchase rate
weekday_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pr_weekday = df.groupby('weekday_label')['purchased'].mean().mul(100).reindex(weekday_order)
axes[0].bar(pr_weekday.index, pr_weekday.values,
            color=sns.color_palette('pastel', 7))
axes[0].set_title('Purchase Rate by Weekday', fontweight='bold')
axes[0].set_ylabel('Purchase Rate (%)')
for i, v in enumerate(pr_weekday.values):
    if not np.isnan(v):
        axes[0].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)

# Seasonal purchase rate
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
pr_season = df.groupby('season_label')['purchased'].mean().mul(100).reindex(season_order)
season_colors = ['#90c97c', '#f4d06f', '#d4843c', '#7ec8e3']
axes[1].bar(pr_season.index, pr_season.values, color=season_colors)
axes[1].set_title('Purchase Rate by Season', fontweight='bold')
axes[1].set_ylabel('Purchase Rate (%)')
for i, v in enumerate(pr_season.values):
    if not np.isnan(v):
        axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Temporal Purchase Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_temporal_patterns.png', bbox_inches='tight')
plt.show()

### 4.9 Correlation Heatmap

In [ ]:
numeric_cols = ['unit_price', 'quantity', 'discount_percent', 'discount_amount',
                'revenue', 'pages_viewed', 'time_on_site_sec', 'added_to_cart',
                'purchased', 'cart_abandoned', 'rating', 'review_helpful_votes']

corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, annot_kws={'size': 9},
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 4.10 New vs Returning Users

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Volume
user_counts = df['user_label'].value_counts()
axes[0].pie(user_counts.values, labels=user_counts.index,
            autopct='%1.1f%%', colors=['#4c72b0', '#dd8452'],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('User Type Distribution', fontweight='bold')

# Purchase rate
pr_user = df.groupby('user_label')['purchased'].mean().mul(100)
axes[1].bar(pr_user.index, pr_user.values, color=['#4c72b0', '#dd8452'], width=0.4)
axes[1].set_title('Purchase Rate: New vs Returning', fontweight='bold')
axes[1].set_ylabel('Purchase Rate (%)')
for i, v in enumerate(pr_user.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontweight='bold')

# Avg revenue
avg_rev_user = df[df['purchased']==1].groupby('user_label')['revenue'].mean()
axes[2].bar(avg_rev_user.index, avg_rev_user.values, color=['#4c72b0', '#dd8452'], width=0.4)
axes[2].set_title('Avg Revenue per Purchase', fontweight='bold')
axes[2].set_ylabel('Avg Revenue (₹)')
for i, v in enumerate(avg_rev_user.values):
    axes[2].text(i, v + 5, f'₹{v:.0f}', ha='center', fontweight='bold')

plt.suptitle('New vs Returning User Behavior', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_user_type.png', bbox_inches='tight')
plt.show()

---
## 5. Feature Engineering <a id='5'></a>

In [ ]:
# ── 5.1 Derived features ─────────────────────────────────────────────────────
df['time_on_site_min']   = df['time_on_site_sec'] / 60
df['effective_price']    = df['unit_price'] * (1 - df['discount_percent'] / 100)
df['total_spend_potential'] = df['effective_price'] * df['quantity']
df['has_discount']       = (df['discount_percent'] > 0).astype(int)
df['is_weekend']         = df['visit_weekday'].isin([5, 6]).astype(int)
df['engagement_score']   = (df['pages_viewed'] * df['time_on_site_min']) / 100

print('✅ New features created:')
new_features = ['time_on_site_min', 'effective_price', 'total_spend_potential',
                'has_discount', 'is_weekend', 'engagement_score']
print(df[new_features].describe().round(2))

In [ ]:
# ── 5.2 Cart behavior summary ────────────────────────────────────────────────
cart_summary = pd.crosstab(df['added_to_cart'], df['purchased'],
                           rownames=['Added to Cart'], colnames=['Purchased'])
cart_summary.index = ['Not Added', 'Added to Cart']
cart_summary.columns = ['Not Purchased', 'Purchased']
print('Cart Behavior vs Purchase Outcome:')
print(cart_summary)
conversion_when_carted = (cart_summary.loc['Added to Cart', 'Purchased'] /
                          cart_summary.loc['Added to Cart'].sum() * 100)
print(f'\nConversion rate when added to cart: {conversion_when_carted:.1f}%')

---
## 6. Customer Segmentation (K-Means Clustering) <a id='6'></a>

In [ ]:
# ── 6.1 Build customer-level RFM-like features ───────────────────────────────
customer_df = df.groupby('customer_id').agg(
    total_sessions   = ('session_id', 'count'),
    total_purchases  = ('purchased', 'sum'),
    total_revenue    = ('revenue', 'sum'),
    avg_pages        = ('pages_viewed', 'mean'),
    avg_time_min     = ('time_on_site_min', 'mean'),
    avg_discount     = ('discount_percent', 'mean'),
    cart_adds        = ('added_to_cart', 'sum'),
    avg_rating       = ('rating', 'mean')
).reset_index()

customer_df['conversion_rate'] = customer_df['total_purchases'] / customer_df['total_sessions']

print(f'Unique customers: {len(customer_df):,}')
customer_df.describe().round(2)

In [ ]:
# ── 6.2 Elbow method to find optimal k ──────────────────────────────────────
cluster_features = ['total_sessions', 'total_purchases', 'total_revenue',
                    'avg_pages', 'avg_time_min', 'conversion_rate']

scaler = StandardScaler()
X_cluster = scaler.fit_transform(customer_df[cluster_features])

inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(list(k_range))
plt.tight_layout()
plt.savefig('fig_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 Fit K-Means with k=4 ─────────────────────────────────────────────────
OPTIMAL_K = 4
km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
customer_df['cluster'] = km_final.fit_predict(X_cluster)

cluster_summary = customer_df.groupby('cluster')[cluster_features + ['conversion_rate']].mean().round(2)
print('=== Cluster Profiles ===')
print(cluster_summary)

In [ ]:
# ── 6.4 Visualise clusters via PCA ───────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)
customer_df['pca1'] = X_pca[:, 0]
customer_df['pca2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
palette = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']
for c in range(OPTIMAL_K):
    mask = customer_df['cluster'] == c
    plt.scatter(customer_df.loc[mask, 'pca1'], customer_df.loc[mask, 'pca2'],
                label=f'Cluster {c}', alpha=0.5, s=18, color=palette[c])
plt.title('Customer Segments — PCA 2D Projection', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig('fig_clusters_pca.png', bbox_inches='tight')
plt.show()
print(f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# ── 6.5 Label clusters with business names ────────────────────────────────────
cluster_labels_map = {
    cluster_summary['total_revenue'].idxmax(): 'High-Value Buyers',
}
remaining = [c for c in range(OPTIMAL_K) if c not in cluster_labels_map]
sorted_by_conv = cluster_summary.loc[remaining, 'conversion_rate'].sort_values(ascending=False)
cluster_labels_map[sorted_by_conv.index[0]] = 'Engaged Browsers'
remaining2 = [c for c in remaining if c != sorted_by_conv.index[0]]
sorted_by_sessions = cluster_summary.loc[remaining2, 'total_sessions'].sort_values(ascending=False)
cluster_labels_map[sorted_by_sessions.index[0]] = 'Casual Visitors'
last = [c for c in remaining2 if c != sorted_by_sessions.index[0]][0]
cluster_labels_map[last] = 'At-Risk Churners'

customer_df['segment'] = customer_df['cluster'].map(cluster_labels_map)
print('Cluster → Segment mapping:')
for k, v in cluster_labels_map.items():
    cnt = (customer_df['cluster'] == k).sum()
    print(f'  Cluster {k} → {v}  ({cnt:,} customers)')

In [ ]:
# Segment profile visualization
seg_summary = customer_df.groupby('segment')[['total_revenue', 'conversion_rate', 'avg_time_min']].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
seg_colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

for ax, col, title, fmt in zip(
    axes,
    ['total_revenue', 'conversion_rate', 'avg_time_min'],
    ['Avg Total Revenue (₹)', 'Avg Conversion Rate', 'Avg Time on Site (min)'],
    ['₹{:.0f}', '{:.2f}', '{:.1f} min']
):
    vals = seg_summary[col].sort_values(ascending=False)
    ax.bar(vals.index, vals.values, color=seg_colors[:len(vals)])
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(vals.index, rotation=15, ha='right', fontsize=9)
    for i, v in enumerate(vals.values):
        ax.text(i, v * 1.02, fmt.format(v), ha='center', fontsize=9)

plt.suptitle('Customer Segment Profiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_segment_profiles.png', bbox_inches='tight')
plt.show()

---
## 7. Purchase Prediction Model (AI) <a id='7'></a>

In [ ]:
# ── 7.1 Prepare features ─────────────────────────────────────────────────────
feature_cols = [
    'device_type', 'user_type', 'marketing_channel', 'product_category',
    'unit_price', 'quantity', 'discount_percent', 'pages_viewed',
    'time_on_site_min', 'added_to_cart', 'rating', 'has_discount',
    'is_weekend', 'engagement_score', 'effective_price', 'visit_month',
    'visit_season', 'visit_weekday'
]

X = df[feature_cols].copy()
y = df['purchased'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Test set     : {X_test.shape[0]:,} samples')
print(f'Class balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# ── 7.2 Train three models ───────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1).mean()
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'auc': auc, 'cv_auc': cv}
    print(f'{name:25s}  Test AUC: {auc:.4f}  |  CV AUC (5-fold): {cv:.4f}')

In [ ]:
# ── 7.3 Classification Reports ───────────────────────────────────────────────
for name, res in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, res['y_pred'],
                                 target_names=['Not Purchased', 'Purchased']))

In [ ]:
# ── 7.4 ROC Curves comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

line_styles = ['-', '--', '-.']
colors_roc  = ['#3b82d4', '#e63946', '#2a9d8f']

for (name, res), ls, color in zip(results.items(), line_styles, colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, ls=ls, color=color, linewidth=2,
                 label=f"{name} (AUC = {res['auc']:.3f})")

axes[0].plot([0,1], [0,1], 'k--', linewidth=1, label='Random Classifier')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Best model confusion matrix
best_name = max(results, key=lambda k: results[k]['auc'])
best_pred = results[best_name]['y_pred']
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Purchased', 'Purchased'])
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')

plt.suptitle('Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_evaluation.png', bbox_inches='tight')
plt.show()
print(f'\n🏆 Best model: {best_name}  (AUC = {results[best_name]["auc"]:.4f})')

In [ ]:
# ── 7.5 Feature Importance (Random Forest) ──────────────────────────────────
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
colors_fi = sns.color_palette('Blues_d', len(importances_sorted))
bars = plt.barh(importances_sorted.index, importances_sorted.values, color=colors_fi)
plt.title('Top 15 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
for bar in bars:
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.6 Model Comparison Summary ────────────────────────────────────────────
from sklearn.metrics import accuracy_score, f1_score

comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, res['y_pred']),
        'F1-Score': f1_score(y_test, res['y_pred']),
        'ROC-AUC': res['auc'],
        'CV AUC (5-fold)': res['cv_auc']
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Model')
print('=== Model Comparison Table ===')
print(comparison_df.round(4))
best = comparison_df['ROC-AUC'].idxmax()
print(f'\n🏆 Best model by AUC: {best}')

---
## 8. Insights & Business Recommendations <a id='8'></a>

In [ ]:
# ── Final summary statistics ──────────────────────────────────────────────────
total_sessions   = len(df)
total_purchases  = df['purchased'].sum()
purchase_rate    = df['purchased'].mean() * 100
total_revenue    = df['revenue'].sum()
avg_order_value  = df[df['purchased']==1]['revenue'].mean()
cart_abandon_rate= df['cart_abandoned'].mean() * 100
top_channel      = df.groupby('channel_label')['purchased'].mean().idxmax()
top_category     = df[df['purchased']==1].groupby('category_label')['revenue'].sum().idxmax()

print('━'*55)
print('       PROJECT SUMMARY — KEY METRICS')
print('━'*55)
print(f'  Total Sessions Analyzed : {total_sessions:,}')
print(f'  Total Purchases         : {total_purchases:,}')
print(f'  Overall Purchase Rate   : {purchase_rate:.2f}%')
print(f'  Total Revenue Generated : ₹{total_revenue:,.2f}')
print(f'  Avg Order Value         : ₹{avg_order_value:,.2f}')
print(f'  Cart Abandonment Rate   : {cart_abandon_rate:.2f}%')
print(f'  Top Marketing Channel   : {top_channel}')
print(f'  Top Revenue Category    : {top_category}')
print(f'  Best ML Model (AUC)     : {best}  ({results[best_name]["auc"]:.4f})')
print('━'*55)

### 📌 Key Insights

1. **Purchase Rate (~13%)** — Only 1 in 8 sessions results in a purchase, indicating significant drop-off at the bottom of the funnel.

2. **Cart is the Strongest Signal** — Sessions where a customer `added_to_cart` have a dramatically higher purchase rate. This is the single most predictive feature.

3. **Discounts Drive Conversions** — Higher discount levels (>20%) correlate with improved purchase rates, especially for price-sensitive categories like Clothing and Sports.

4. **Email & Referral Channels Outperform** — These channels bring users with higher purchase intent compared to Paid and Organic.

5. **Engagement Matters** — Customers who view more pages and spend more time on the site are more likely to purchase. Engagement score is a reliable proxy.

6. **Returning Users are More Valuable** — Returning users have a slightly higher purchase rate and generate higher average revenue per transaction.

7. **Seasonal Peaks** — Autumn/Winter months show higher session volumes and purchase rates (holiday effect).

8. **Electronics Leads Revenue** — Despite not having the highest volume, Electronics contributes the most to total revenue.

---

### 💼 Business Recommendations

| # | Recommendation | Expected Impact |
|---|---|---|
| 1 | **Cart Abandonment Recovery** — Trigger email/push within 1 hour of cart abandonment | +15–25% recovered conversions |
| 2 | **Personalized Discounts for At-Risk Churners** — Offer targeted 15–20% discounts to churner segment | Re-engage high-value lost customers |
| 3 | **Invest in Email Marketing** — Scale budget for Email channel (highest purchase rate) | Improve marketing ROI |
| 4 | **Mobile Optimization** — Mobile accounts for the largest session share; improve UX for conversion parity | +5–10% mobile conversion uplift |
| 5 | **Deploy Predictive Model** — Use the trained Random Forest to score sessions in real time and trigger interventions | Proactive conversion optimization |
| 6 | **Loyalty Program for Returning Users** — Build rewards to increase repeat visit frequency | Increase customer lifetime value |
| 7 | **Seasonal Campaigns** — Pre-load inventory and campaigns for Q4 (highest revenue period) | Capture seasonal demand |
| 8 | **Category Expansion in Electronics** — Given highest revenue, deepen catalogue and improve filters | Expand leading revenue category |

In [ ]:
print('🎉 Analysis Complete!')
print('All visualizations saved as PNG files in the project directory.')
print('\nFiles generated:')
import glob as glb
for f in sorted(glb.glob('fig_*.png')):
    print(f'  📊 {f}')